# CogAttention — Visual Inattentional Blindness

**Track:** Attention — Visual Inattentional Blindness (Multimodal VLM Extension)
**Benchmark:** CogAttention v1.0
**Tasks:** visual_inattentional

---

## Methodology

Based on **Simons & Chabris (1999)** gorilla experiment. Subjects focused on a
counting task miss an unexpected stimulus embedded in the scene.

Tests whether VLMs exhibit **inattentional blindness**: does task-directed focus
cause them to miss anomalous objects in the visual field?

### Cognitive Science Grounding

- **Inattentional blindness** (Simons & Chabris, 1999): failure to notice unexpected
  stimuli when attention is engaged elsewhere.
- **Selective attention** (Cherry, 1953; Broadbent, 1958): filtering relevant from
  irrelevant stimuli under cognitive load.
- **Divided attention** (Kahneman, 1973): performing concurrent tasks degrades
  detection of peripheral stimuli.

### Difficulty Scaling

| Level    | Shapes | Unexpected Size | Saliency  | Stimulus Type   |
|----------|--------|-----------------|-----------|-----------------|
| Easy     | 8-12   | Large           | High      | Star            |
| Medium   | 15-20  | Medium          | Medium    | Arrow           |
| Hard     | 25-30  | Small           | Low       | Cross           |
| Expert   | 35-40  | Tiny            | Very Low  | Dot Pattern     |
| Frontier | 45-50  | Subtle          | Ultra Low | Gradient Patch  |

50% of items contain an unexpected stimulus; 50% do not.
All images generated procedurally with PIL — zero data leakage.

### Scoring

SDK assertion pass rate = per-element accuracy. Two assertions per item:
1. Correct count of target shapes.
2. Detection (or correct non-detection) of unexpected stimulus.

---

`<!-- COGATTENTION-BENCH-CANARY-7E5E99CE7F54 -->`

In [ ]:
# ======================================================================
# Cell 2: Imports + Inline Helpers + Generator + Assertions
# CogAttention — Visual Inattentional Blindness
# ======================================================================

import kaggle_benchmarks as kbench

import json
import re
import random
import base64
import math
from io import BytesIO

try:
    from PIL import Image, ImageDraw, ImageFont
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

# ── Constants ──────────────────────────────────────────────────────────

ITEMS_PER_DIFFICULTY = 30
DIFFICULTY_LEVELS = ["Easy", "Medium", "Hard", "Expert", "Frontier"]

SHAPE_TYPES = ["circle", "square", "triangle"]
SHAPE_COLORS = {
    "red": (220, 40, 40), "blue": (40, 80, 220), "green": (40, 180, 60),
    "yellow": (220, 200, 40), "orange": (230, 130, 30), "purple": (140, 40, 180),
}
COLOR_NAMES = list(SHAPE_COLORS.keys())

UNEXPECTED_TYPES = {
    "Easy": "star", "Medium": "arrow", "Hard": "cross",
    "Expert": "dot_pattern", "Frontier": "gradient_patch",
}

DIFFICULTY_CONFIG = {
    "Easy": {"shape_count": (8, 12), "unexpected_size": "large", "saliency": "high"},
    "Medium": {"shape_count": (15, 20), "unexpected_size": "medium", "saliency": "medium"},
    "Hard": {"shape_count": (25, 30), "unexpected_size": "small", "saliency": "low"},
    "Expert": {"shape_count": (35, 40), "unexpected_size": "tiny", "saliency": "very_low"},
    "Frontier": {"shape_count": (45, 50), "unexpected_size": "subtle", "saliency": "ultra_low"},
}

IMG_WIDTH = 600
IMG_HEIGHT = 500

# ── Drawing Functions ─────────────────────────────────────────────────

def _draw_shape(draw, shape_type, x, y, size, color):
    """Draw a basic shape on the image."""
    if shape_type == "circle":
        draw.ellipse([x, y, x + size, y + size], fill=color)
    elif shape_type == "square":
        draw.rectangle([x, y, x + size, y + size], fill=color)
    elif shape_type == "triangle":
        points = [
            (x + size // 2, y),
            (x, y + size),
            (x + size, y + size),
        ]
        draw.polygon(points, fill=color)


def _draw_star(draw, cx, cy, size, color):
    """Draw a 5-pointed star."""
    points = []
    for i in range(10):
        angle = math.pi / 2 + i * math.pi / 5
        r = size if i % 2 == 0 else size * 0.4
        px = cx + int(r * math.cos(angle))
        py = cy - int(r * math.sin(angle))
        points.append((px, py))
    draw.polygon(points, fill=color)


def _draw_arrow(draw, cx, cy, size, color):
    """Draw an arrow shape."""
    half = size // 2
    # Arrow body
    draw.rectangle([cx - half, cy - size // 6, cx + half // 2, cy + size // 6], fill=color)
    # Arrow head
    points = [
        (cx + half // 2, cy - half),
        (cx + half, cy),
        (cx + half // 2, cy + half),
    ]
    draw.polygon(points, fill=color)


def _draw_cross(draw, cx, cy, size, color):
    """Draw a cross/plus shape."""
    t = max(size // 4, 2)
    draw.rectangle([cx - t, cy - size, cx + t, cy + size], fill=color)
    draw.rectangle([cx - size, cy - t, cx + size, cy + t], fill=color)


def _draw_dot_pattern(draw, cx, cy, size, color, rng):
    """Draw a cluster of small dots."""
    for _ in range(8):
        dx = rng.randint(-size, size)
        dy = rng.randint(-size, size)
        r = max(size // 6, 2)
        draw.ellipse([cx + dx - r, cy + dy - r, cx + dx + r, cy + dy + r], fill=color)


def _draw_gradient_patch(draw, img, cx, cy, size, color):
    """Draw a subtle gradient patch."""
    for dy in range(-size, size + 1):
        for dx in range(-size, size + 1):
            px, py = cx + dx, cy + dy
            if 0 <= px < IMG_WIDTH and 0 <= py < IMG_HEIGHT:
                dist = math.sqrt(dx * dx + dy * dy)
                if dist <= size:
                    alpha = 1.0 - (dist / size)
                    bg = img.getpixel((px, py))
                    blended = tuple(
                        int(bg[c] * (1 - alpha * 0.5) + color[c] * alpha * 0.5)
                        for c in range(3)
                    )
                    draw.point((px, py), fill=blended)


def _get_unexpected_color(saliency, target_color_name, bg_color, rng):
    """Get color for unexpected stimulus based on saliency level."""
    if saliency == "high":
        contrasting = [c for c in COLOR_NAMES if c != target_color_name]
        chosen = rng.choice(contrasting)
        return SHAPE_COLORS[chosen]
    elif saliency == "medium":
        base = SHAPE_COLORS[rng.choice(COLOR_NAMES)]
        return tuple(min(255, c + rng.randint(-30, 30)) for c in base)
    elif saliency == "low":
        return tuple(min(255, bg_color[c] + rng.randint(30, 60)) for c in range(3))
    elif saliency == "very_low":
        return tuple(min(255, bg_color[c] + rng.randint(15, 35)) for c in range(3))
    else:  # ultra_low
        return tuple(min(255, bg_color[c] + rng.randint(5, 20)) for c in range(3))


def _get_unexpected_size(size_label):
    """Get pixel size for unexpected stimulus."""
    sizes = {
        "large": 40,
        "medium": 28,
        "small": 18,
        "tiny": 10,
        "subtle": 12,
    }
    return sizes.get(size_label, 20)


# ── Main Image Generator ──────────────────────────────────────────────

def _generate_inattentional_image(
    target_color_name,
    target_shape,
    n_shapes,
    unexpected_present,
    unexpected_type,
    unexpected_size_label,
    saliency,
    rng,
):
    """Generate a scene image with shapes and optionally an unexpected stimulus.

    Returns:
        (base64_png, actual_target_count, unexpected_description)
    """
    if not PIL_AVAILABLE:
        count = rng.randint(3, 8)
        desc = f"{unexpected_type}" if unexpected_present else ""
        return f"[IMAGE_PLACEHOLDER: shapes={n_shapes}]", count, desc

    bg_color = (230, 230, 230)
    img = Image.new("RGB", (IMG_WIDTH, IMG_HEIGHT), color=bg_color)
    draw = ImageDraw.Draw(img)

    target_rgb = SHAPE_COLORS[target_color_name]
    target_count = 0

    # Place shapes avoiding too much overlap
    shape_size = max(18, min(30, 500 // int(math.sqrt(n_shapes))))
    positions = []

    for i in range(n_shapes):
        x = rng.randint(10, IMG_WIDTH - shape_size - 10)
        y = rng.randint(10, IMG_HEIGHT - shape_size - 10)
        positions.append((x, y))

        # Decide if this shape is the target type+color
        is_target = rng.random() < 0.25  # ~25% are targets
        if is_target:
            _draw_shape(draw, target_shape, x, y, shape_size, target_rgb)
            target_count += 1
        else:
            # Random non-target (different shape or different color)
            if rng.random() < 0.5:
                # Different color, same shape
                other_colors = [c for c in COLOR_NAMES if c != target_color_name]
                other_color = SHAPE_COLORS[rng.choice(other_colors)]
                _draw_shape(draw, target_shape, x, y, shape_size, other_color)
            else:
                # Different shape, any color
                other_shapes = [s for s in SHAPE_TYPES if s != target_shape]
                other_shape = rng.choice(other_shapes)
                any_color = SHAPE_COLORS[rng.choice(COLOR_NAMES)]
                _draw_shape(draw, other_shape, x, y, shape_size, any_color)

    # Ensure at least 1 target
    if target_count == 0:
        x, y = positions[0] if positions else (IMG_WIDTH // 2, IMG_HEIGHT // 2)
        _draw_shape(draw, target_shape, x, y, shape_size, target_rgb)
        target_count = 1

    # Draw unexpected stimulus if present
    unexpected_description = ""
    if unexpected_present:
        ux_size = _get_unexpected_size(unexpected_size_label)
        ux_color = _get_unexpected_color(saliency, target_color_name, bg_color, rng)
        # Place in a semi-random but visible location
        ux = rng.randint(IMG_WIDTH // 4, 3 * IMG_WIDTH // 4)
        uy = rng.randint(IMG_HEIGHT // 4, 3 * IMG_HEIGHT // 4)

        if unexpected_type == "star":
            _draw_star(draw, ux, uy, ux_size, ux_color)
            unexpected_description = "a star shape"
        elif unexpected_type == "arrow":
            _draw_arrow(draw, ux, uy, ux_size, ux_color)
            unexpected_description = "an arrow shape"
        elif unexpected_type == "cross":
            _draw_cross(draw, ux, uy, ux_size, ux_color)
            unexpected_description = "a cross/plus shape"
        elif unexpected_type == "dot_pattern":
            _draw_dot_pattern(draw, ux, uy, ux_size, ux_color, rng)
            unexpected_description = "a cluster of dots"
        elif unexpected_type == "gradient_patch":
            _draw_gradient_patch(draw, img, ux, uy, ux_size, ux_color)
            unexpected_description = "a subtle gradient patch"

    # Convert to base64
    buffered = BytesIO()
    img.save(buffered, format="PNG")
    img_b64 = base64.b64encode(buffered.getvalue()).decode("ascii")

    return img_b64, target_count, unexpected_description


# ── Dataset Generator ─────────────────────────────────────────────────

def generate_visual_inattentional_dataset(seed=2026):
    """Generate full Visual Inattentional Blindness dataset across all difficulty tiers."""
    dataset = []
    idx = 0
    for diff in DIFFICULTY_LEVELS:
        config = DIFFICULTY_CONFIG[diff]
        for i in range(ITEMS_PER_DIFFICULTY):
            rng = random.Random(seed + idx * 1049 + DIFFICULTY_LEVELS.index(diff) * 3907)

            target_color = rng.choice(COLOR_NAMES)
            target_shape = rng.choice(SHAPE_TYPES)
            n_shapes = rng.randint(*config["shape_count"])

            # 50% have unexpected stimulus
            unexpected_present = (idx % 2 == 0)
            unexpected_type = UNEXPECTED_TYPES[diff]

            img_b64, target_count, unexpected_desc = _generate_inattentional_image(
                target_color_name=target_color,
                target_shape=target_shape,
                n_shapes=n_shapes,
                unexpected_present=unexpected_present,
                unexpected_type=unexpected_type,
                unexpected_size_label=config["unexpected_size"],
                saliency=config["saliency"],
                rng=rng,
            )

            prompt = (
                f"Look at the image carefully.\n\n"
                f"PRIMARY TASK: Count the number of {target_color} {target_shape}s in the image.\n\n"
                f"SECONDARY TASK: Did you notice anything unusual or out-of-place in the image? "
                f"If so, describe it briefly. If not, say 'nothing unusual'.\n\n"
                f"ANSWER:\n"
                f"1. Count of {target_color} {target_shape}s: [number]\n"
                f"2. Unusual observation: [description or 'nothing unusual']"
            )

            gold_answer = {
                "count": target_count,
                "unexpected_present": unexpected_present,
                "unexpected_description": unexpected_desc if unexpected_present else "",
            }

            task_id = f"visual_inattentional_{diff.lower()}_{idx:03d}"

            dataset.append({
                "task_id": task_id,
                "task_type": "visual_inattentional",
                "difficulty": diff,
                "prompt": prompt,
                "gold": gold_answer,
                "image": img_b64,
                "metadata": {
                    "target_color": target_color,
                    "target_shape": target_shape,
                    "n_shapes": n_shapes,
                    "target_count": target_count,
                    "unexpected_present": unexpected_present,
                    "unexpected_type": unexpected_type if unexpected_present else None,
                    "unexpected_description": unexpected_desc,
                    "saliency": config["saliency"],
                },
            })
            idx += 1
    return dataset


# ── Assertion Function ────────────────────────────────────────────────

def run_assertions_visual_inattentional(response, gold, kbench):
    # Check count — allow +/- 1 tolerance since counting shapes in a busy image is hard
    count = gold["count"]
    tolerance = 1
    alternatives = [str(n) for n in range(max(0, count - tolerance), count + tolerance + 1)]
    alt_pattern = "|".join(rf"\b{a}\b" for a in alternatives)
    pattern_count = rf"(?i)(?:{alt_pattern})"
    kbench.assertions.assert_contains_regex(
        pattern_count, response,
        expectation=f"Count of target shapes should be approximately {count}"
    )
    # Check unexpected stimulus detection
    if gold["unexpected_present"]:
        desc = gold["unexpected_description"]
        words = [w for w in desc.split() if len(w) > 2 and w.lower() not in {"the", "and", "was"}]
        # Use multiple keywords for broader matching
        keywords = [re.escape(w) for w in words] if words else [re.escape(desc.split()[-1]) if desc else "unusual"]
        # Also add generic detection words
        keywords.extend(["unusual", "unexpected", "different", "noticed", "odd", "strange", "stand out", "unique"])
        pattern_detect = rf"(?i)(?:{'|'.join(keywords)})"
        kbench.assertions.assert_contains_regex(
            pattern_detect, response,
            expectation=f"Should detect unexpected stimulus: {desc}"
        )


print("CogAttention Visual Inattentional Blindness helpers loaded")
print(f"Task types: ['visual_inattentional']")

In [ ]:
# ======================================================================
# Cell 3: Task Definition + Dataset Generation
# ======================================================================


@kbench.task(name="cogattention_visual_inattentional")
def cogattention_visual_inattentional(llm, prompt: str, gold_json: str, image_data: str, task_id: str, difficulty: str):
    """CogAttention visual inattentional blindness task."""
    full_prompt = prompt + f"\n\n[Image (base64 PNG)]: {image_data}"
    response = llm.prompt(full_prompt)
    gold = json.loads(gold_json)
    run_assertions_visual_inattentional(response, gold, kbench)


print("Generating Visual Inattentional Blindness dataset...")
raw_dataset = generate_visual_inattentional_dataset(seed=2026)
DATASET = []
for inst in raw_dataset:
    DATASET.append({
        "task_id": inst["task_id"],
        "task_type": inst["task_type"],
        "difficulty": inst["difficulty"],
        "prompt": inst["prompt"],
        "gold_json": json.dumps(inst["gold"]),
        "image_data": inst["image"],
    })
print(f"Generated {len(DATASET)} items")

In [ ]:
# ======================================================================
# Cell 4: Execution Loop
# ======================================================================

TASK_DISPATCH = {"visual_inattentional": cogattention_visual_inattentional}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        image_data=item["image_data"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Visual Inattentional Blindness")